# Gold layer

Builds the gold tables and the KPI views in Athena from the silver tables, and checks them against the source files.

| Table | One row per | Built by |
|---|---|---|
| `dim_patient`, `dim_facility`, `dim_practitioner` | patient / hospital / doctor | `MERGE` (upsert) |
| `dim_date` | calendar day (1900-2030) | one-time `CREATE TABLE AS` |
| `fact_encounter` | visit or stay | `MERGE` (upsert) |

All gold tables are Apache Iceberg tables. The SQL lives in `sql/gold/` (one file per table, merge and view).
Run the cells top to bottom; every step can be repeated safely.

## 1. Setup

In [1]:
import json
import sys
import time
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

# Works whether Jupyter starts in the project root or in notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "infra"))

# aws() runs the AWS CLI; run_athena / run_sql_file run SQL in the pf-wg workgroup (see infra/aws_helpers.py)
from aws_helpers import SETTINGS, aws, run_athena, run_glue_job, run_sql_file, whoami

SILVER = SETTINGS["glue_databases"]["silver"]
GOLD = SETTINGS["glue_databases"]["gold"]
INGEST_DATES = [f"2026-11-{n:02d}" for n in range(1, SETTINGS["number_of_batches"] + 1)]


def scalar(sql):
    return run_athena(sql)["rows"][0][0]


def count(sql):
    return int(scalar(sql))


print("identity   :", whoami())
print("ingest dates:", INGEST_DATES)

identity   : user/Ahmed
ingest dates: ['2026-11-01', '2026-11-02', '2026-11-03', '2026-11-04']


## 2. Silver must hold every batch
The gold merges read silver one `ingest_date` at a time. Batches 3 and 4 have not been processed yet, so any batch missing from
silver is ingested by the Lambda and transformed by the Glue job first (the same steps the pipeline will run by itself later).

In [2]:
def silver_dates():
    return {row[0] for row in run_athena(f"SELECT DISTINCT ingest_date FROM {SILVER}.patient")["rows"]}


def ingest_and_transform(ingest_date):
    batch = int(ingest_date[-2:])
    out_file = ROOT / "build" / "invoke_result.json"
    meta = json.loads(aws("lambda", "invoke", "--function-name", SETTINGS["lambda_name"],
                          "--payload", json.dumps({"batch": batch}), "--cli-binary-format", "raw-in-base64-out",
                          str(out_file.relative_to(ROOT))).stdout)
    assert "FunctionError" not in meta, out_file.read_text()
    run = run_glue_job(SETTINGS["glue_job_name"], {"--ingest_date": ingest_date})
    assert run["state"] == "SUCCEEDED", run["error"]


for ingest_date in INGEST_DATES:
    if ingest_date not in silver_dates():
        print(f"\nsilver is missing {ingest_date}: ingesting and transforming batch {int(ingest_date[-2:])}")
        ingest_and_transform(ingest_date)

assert silver_dates() == set(INGEST_DATES), silver_dates()
print("\nOK  silver holds", sorted(silver_dates()))


OK  silver holds ['2026-11-01', '2026-11-02', '2026-11-03', '2026-11-04']


## 3. Create the gold tables
`CREATE TABLE IF NOT EXISTS` for the four merged tables. The date dimension is built once with `CREATE TABLE AS`
(`sequence()` creates the days, `UNNEST` turns them into rows), only if it does not exist yet.
In `CREATE TABLE`, Athena uses Hive type names (`string`, `int`); in queries it uses Trino names (`varchar`, `integer`).

In [3]:
for table in ("dim_patient", "dim_facility", "dim_practitioner", "fact_encounter"):
    run_sql_file(f"sql/gold/ddl/{table}.sql")
    print("OK  table", table)

DATE_FIRST, DATE_LAST = date(1900, 1, 1), date(2030, 12, 31)
EXPECTED_DAYS = (DATE_LAST - DATE_FIRST).days + 1

existing = {row[0] for row in run_athena(f"SHOW TABLES IN {GOLD}")["rows"]}
date_key_type = scalar(
    "SELECT data_type FROM information_schema.columns "
    f"WHERE table_schema = '{GOLD}' AND table_name = 'dim_date' AND column_name = 'date_key'") if "dim_date" in existing else None

if "dim_date" in existing and date_key_type == "date" and count(f"SELECT COUNT(*) FROM {GOLD}.dim_date") == EXPECTED_DAYS:
    print("OK  table dim_date already exists with", EXPECTED_DAYS, "days")
else:
    if "dim_date" in existing:
        print("dim_date has a different range or column type: dropping it to rebuild")
        run_athena(f"DROP TABLE {GOLD}.dim_date")
    run_sql_file("sql/gold/ddl/dim_date.sql")
    print("OK  table dim_date created with", EXPECTED_DAYS, "days")

OK  table dim_patient


OK  table dim_facility


OK  table dim_practitioner


OK  table fact_encounter


OK  table dim_date already exists with 47847 days


## 4. Load the tables with MERGE, batch by batch
For every batch: the three dimensions first, then the fact, so the dimensions exist before the facts that point to them.
Each `MERGE` reads only the silver rows of that `ingest_date` (passed as the `?` parameter), inserts new keys and updates known ones.

In [4]:
MERGE_ORDER = ["merge_dim_patient", "merge_dim_facility", "merge_dim_practitioner", "merge_fact_encounter"]


def merge_batch(ingest_date):
    for name in MERGE_ORDER:
        started = time.time()
        run_sql_file(f"sql/gold/merge/{name}.sql", ingest_date=ingest_date)
        print(f"  {ingest_date}  {name:<24} {time.time() - started:4.0f} s")


for ingest_date in INGEST_DATES:
    merge_batch(ingest_date)

  2026-11-01  merge_dim_patient          10 s


  2026-11-01  merge_dim_facility          8 s


  2026-11-01  merge_dim_practitioner      8 s


  2026-11-01  merge_fact_encounter       13 s


  2026-11-02  merge_dim_patient           8 s


...

  2026-11-03  merge_fact_encounter       11 s


  2026-11-04  merge_dim_patient          12 s


  2026-11-04  merge_dim_facility          7 s


  2026-11-04  merge_dim_practitioner      5 s


  2026-11-04  merge_fact_encounter       11 s


## 5. Check the row counts
Each gold table must hold exactly the distinct keys that exist in silver.

In [5]:
def gold_checks():
    return {
        "dim_patient": (count(f"SELECT COUNT(*) FROM {GOLD}.dim_patient"),
                        count(f"SELECT COUNT(DISTINCT patient_id) FROM {SILVER}.patient")),
        "dim_facility": (count(f"SELECT COUNT(*) FROM {GOLD}.dim_facility"),
                         count(f"SELECT COUNT(DISTINCT organization_id) FROM {SILVER}.organization")),
        "dim_practitioner": (count(f"SELECT COUNT(*) FROM {GOLD}.dim_practitioner"),
                             count(f"SELECT COUNT(DISTINCT npi) FROM {SILVER}.practitioner")),
        "fact_encounter": (count(f"SELECT COUNT(*) FROM {GOLD}.fact_encounter"),
                           count(f"SELECT COUNT(DISTINCT encounter_id) FROM {SILVER}.encounter")),
    }


counts = gold_checks()
for table, (gold, silver) in counts.items():
    assert gold == silver, f"{table}: gold {gold} vs silver {silver}"
    print(f"OK  {table:<17} {gold:>6} rows = distinct keys in silver")

assert count(f"SELECT COUNT(*) FROM {GOLD}.dim_date") == EXPECTED_DAYS
print(f"OK  dim_date          {EXPECTED_DAYS:>6} rows = every day from {DATE_FIRST} to {DATE_LAST}")

OK  dim_patient          137 rows = distinct keys in silver
OK  dim_facility         521 rows = distinct keys in silver
OK  dim_practitioner     521 rows = distinct keys in silver
OK  fact_encounter      4586 rows = distinct keys in silver


OK  dim_date           47847 rows = every day from 1900-01-01 to 2030-12-31


In [6]:
KEYS = {"dim_patient": "patient_id", "dim_facility": "organization_id", "dim_practitioner": "npi",
        "fact_encounter": "encounter_id", "dim_date": "date_key"}
for table, key in KEYS.items():
    duplicates = count(f"SELECT COUNT(*) - COUNT(DISTINCT {key}) FROM {GOLD}.{table}")
    nulls = count(f"SELECT COUNT(*) FROM {GOLD}.{table} WHERE {key} IS NULL")
    assert duplicates == 0 and nulls == 0, f"{table}: {duplicates} duplicate and {nulls} null keys"
    print(f"OK  {table:<17} no duplicate or null {key}")

OK  dim_patient       no duplicate or null patient_id


OK  dim_facility      no duplicate or null organization_id


OK  dim_practitioner  no duplicate or null npi


OK  fact_encounter    no duplicate or null encounter_id


OK  dim_date          no duplicate or null date_key


## 6. Check the values against the source files
The expected values are computed with plain Python from the original FHIR files, independently of Spark and SQL.

In [7]:
def utc(iso_text):
    return datetime.fromisoformat(iso_text).astimezone(timezone.utc)


source_encounters, source_conditions = {}, []
for path in sorted((ROOT / "data" / "landing").glob("batch_*/*.json")):
    if path.name.startswith(("hospitalInformation", "practitionerInformation")):
        continue
    for entry in json.loads(path.read_text(encoding="utf-8"))["entry"]:
        resource = entry["resource"]
        if resource["resourceType"] == "Encounter":
            source_encounters[resource["id"]] = resource
        elif resource["resourceType"] == "Condition":
            source_conditions.append(resource)

by_class = {}
for encounter in source_encounters.values():
    by_class.setdefault(encounter["class"]["code"], []).append(encounter)
print({cls: len(items) for cls, items in sorted(by_class.items())}, "| total", len(source_encounters))
assert len(source_encounters) == counts["fact_encounter"][0]

{'AMB': 4313, 'EMER': 127, 'HH': 14, 'IMP': 114, 'VR': 18} | total 4586


In [8]:
# One encounter of each of three classes, field by field
def primary_condition(encounter_id):
    candidates = [(utc(c["onsetDateTime"]), c["id"], c["code"]["coding"][0]["code"]) for c in source_conditions
                  if c["encounter"]["reference"] == f"urn:uuid:{encounter_id}"]
    return min(candidates)[2] if candidates else None


for cls in ("IMP", "EMER", "AMB"):
    source = by_class[cls][0]
    start, end = utc(source["period"]["start"]), utc(source["period"]["end"])
    expected = [source["id"], cls, start.strftime("%Y-%m-%d"), end.strftime("%Y-%m-%d"),
                round((end - start).total_seconds() / 3600, 6), str(cls == "IMP").lower(), str(cls == "EMER").lower(),
                source["reasonCode"][0]["coding"][0]["code"] if "reasonCode" in source else None,
                primary_condition(source["id"])]
    row = run_athena(
        "SELECT encounter_id, encounter_class, CAST(start_date AS varchar), CAST(end_date AS varchar), "
        "round(los_hours, 6), CAST(is_inpatient AS varchar), CAST(is_emergency AS varchar), reason_code, "
        f"primary_condition_code FROM {GOLD}.fact_encounter WHERE encounter_id = '{source['id']}'")["rows"][0]
    actual = [row[0], row[1], row[2], row[3], round(float(row[4]), 6), row[5], row[6], row[7], row[8]]
    assert actual == expected, (actual, expected)
    print(f"OK  {cls:<5} {source['id']}  los {actual[4]:>9} h  reason {actual[7]}  primary condition {actual[8]}")

OK  IMP   2c588417-7537-a68c-0ec1-f4fa3d2d5f6e  los 76.923611 h  reason 698306007  primary condition 161665007


OK  EMER  2ae1475c-5608-a2c1-017e-3b144b6f8851  los       1.0 h  reason 124171000119105  primary condition 124171000119105


OK  AMB   f75caf9c-ba51-5e74-b210-6dc056eeb080  los      0.25 h  reason 36971009  primary condition 40055000


In [9]:
# Age is calculated on the day the merge runs: compare with plain Python
today = date.fromisoformat(scalar("SELECT CAST(current_date AS varchar)"))
patient_id, birth_date, age_years, age_group = run_athena(
    f"SELECT patient_id, CAST(birth_date AS varchar), age_years, age_group FROM {GOLD}.dim_patient "
    "WHERE is_deceased = false ORDER BY patient_id LIMIT 1")["rows"][0]
born = date.fromisoformat(birth_date)
expected_age = today.year - born.year - ((today.month, today.day) < (born.month, born.day))
expected_group = "0-17" if expected_age < 18 else "18-34" if expected_age < 35 else "35-49" if expected_age < 50 else "50-64" if expected_age < 65 else "65+"
assert (int(age_years), age_group) == (expected_age, expected_group), (age_years, age_group, expected_age)
print(f"OK  patient born {birth_date}: age {age_years} ({age_group}) as of {today}")

deceased = count(f"SELECT COUNT(*) FROM {GOLD}.dim_patient WHERE is_deceased")
assert deceased == count(f"SELECT COUNT(*) FROM {SILVER}.patient WHERE deceased_datetime IS NOT NULL")
print(f"OK  {deceased} deceased patients flagged")

OK  patient born 2018-02-17: age 8 (0-17) as of 2026-10-26


OK  14 deceased patients flagged


In [10]:
# The fact table is partitioned by the year of start_ts (an Iceberg partition transform)
partitions = run_athena(f'SELECT partition FROM "{GOLD}"."fact_encounter$partitions"')["rows"]
years_in_silver = count(f"SELECT COUNT(DISTINCT year(start_ts)) FROM {SILVER}.encounter")
years_in_gold = count(f"SELECT COUNT(DISTINCT year(start_ts)) FROM {GOLD}.fact_encounter")
assert len(partitions) == years_in_silver == years_in_gold, (len(partitions), years_in_silver, years_in_gold)
print(f"OK  fact_encounter holds {len(partitions)} yearly partitions, one per year with visits; example: {partitions[0]}")

# Note: the metadata table's record_count is NOT the number of rows. An Iceberg MERGE writes new row versions and
# marks the old ones as deleted; the old versions stay in the files until the table is compacted (OPTIMIZE / VACUUM).
stored = sum(int(row[0]) for row in run_athena(f'SELECT record_count FROM "{GOLD}"."fact_encounter$partitions"')["rows"])
live = counts["fact_encounter"][0]
print(f"    rows a query sees: {live} | row versions stored in the data files: {stored}")
assert stored >= live

OK  fact_encounter holds 82 yearly partitions, one per year with visits; example: ['{start_ts_year=15}']


    rows a query sees: 4586 | row versions stored in the data files: 22906


## 7. Run the merges again: no duplicates
Merging a batch that is already loaded must change nothing: the same row counts and the same keys (every row is matched and updated, none inserted).

In [11]:
before = gold_checks()
for ingest_date in INGEST_DATES[:2]:
    merge_batch(ingest_date)

after = gold_checks()
assert after == before, (before, after)
for key_table, key in KEYS.items():
    assert count(f"SELECT COUNT(*) - COUNT(DISTINCT {key}) FROM {GOLD}.{key_table}") == 0
print("OK  second run of batches 1 and 2: same row counts, still no duplicate keys")

  2026-11-01  merge_dim_patient           9 s


  2026-11-01  merge_dim_facility         10 s


  2026-11-01  merge_dim_practitioner      6 s


  2026-11-01  merge_fact_encounter        9 s


  2026-11-02  merge_dim_patient           8 s


  2026-11-02  merge_dim_facility          5 s


  2026-11-02  merge_dim_practitioner      5 s


  2026-11-02  merge_fact_encounter       10 s


OK  second run of batches 1 and 2: same row counts, still no duplicate keys


## 8. Every fact must point to a dimension row

In [12]:
orphans = {
    "encounters without a patient": f"SELECT COUNT(*) FROM {GOLD}.fact_encounter f LEFT JOIN {GOLD}.dim_patient d ON f.patient_id = d.patient_id WHERE d.patient_id IS NULL",
    "encounters without a facility": f"SELECT COUNT(*) FROM {GOLD}.fact_encounter f LEFT JOIN {GOLD}.dim_facility d ON f.organization_id = d.organization_id WHERE d.organization_id IS NULL",
    "encounters without a doctor": f"SELECT COUNT(*) FROM {GOLD}.fact_encounter f LEFT JOIN {GOLD}.dim_practitioner d ON f.practitioner_npi = d.npi WHERE d.npi IS NULL",
    "encounters without a start date": f"SELECT COUNT(*) FROM {GOLD}.fact_encounter f LEFT JOIN {GOLD}.dim_date d ON f.start_date = d.date_key WHERE d.date_key IS NULL",
}
for label, sql in orphans.items():
    assert count(sql) == 0, label
    print("OK  0", label)

OK  0 encounters without a patient


OK  0 encounters without a facility


OK  0 encounters without a doctor


OK  0 encounters without a start date


## 9. The KPI views
A view stores a query, not data: it always shows the current state of `fact_encounter`.

In [13]:
VIEWS = ["vw_daily_admissions_discharges", "vw_avg_los_by_facility_month", "vw_readmission_30d",
         "vw_ed_volume_by_hour_dow", "vw_top_reasons"]
for view in VIEWS:
    run_sql_file(f"sql/gold/views/{view}.sql")
    rows = count(f"SELECT COUNT(*) FROM {GOLD}.{view}")
    assert rows > 0, f"{view} is empty"
    print(f"OK  {view:<34} {rows:>6} rows")

OK  vw_daily_admissions_discharges        213 rows


OK  vw_avg_los_by_facility_month          106 rows


OK  vw_readmission_30d                    106 rows


OK  vw_ed_volume_by_hour_dow               78 rows


OK  vw_top_reasons                        109 rows


In [14]:
inpatient = [e for e in source_encounters.values() if e["class"]["code"] == "IMP"]
emergency = [e for e in source_encounters.values() if e["class"]["code"] == "EMER"]

# admissions and discharges: one of each per inpatient stay
totals = run_athena(f"SELECT SUM(admissions), SUM(discharges) FROM {GOLD}.vw_daily_admissions_discharges")["rows"][0]
assert [int(x) for x in totals] == [len(inpatient), len(inpatient)]
print(f"OK  admissions and discharges: {totals[0]} each = inpatient stays in the source")

# emergency visits
assert count(f"SELECT SUM(emergency_visits) FROM {GOLD}.vw_ed_volume_by_hour_dow") == len(emergency)
print(f"OK  emergency visits: {len(emergency)} = EMER encounters in the source")

# average length of stay: stay-weighted average of the view equals the plain average over all stays
expected_avg = sum((utc(e["period"]["end"]) - utc(e["period"]["start"])).total_seconds() / 86400 for e in inpatient) / len(inpatient)
view_avg = float(scalar(f"SELECT SUM(avg_los_days * inpatient_stays) / SUM(inpatient_stays) FROM {GOLD}.vw_avg_los_by_facility_month"))
assert abs(view_avg - expected_avg) < 1e-6, (view_avg, expected_avg)
print(f"OK  average stay: {view_avg:.3f} days = source")

# top reason
reasons = {}
for e in source_encounters.values():
    if "reasonCode" in e:
        code = e["reasonCode"][0]["coding"][0]["code"]
        reasons[code] = reasons.get(code, 0) + 1
top_code, top_count = max(reasons.items(), key=lambda item: item[1])
row = run_athena(f"SELECT reason_code, encounters FROM {GOLD}.vw_top_reasons WHERE rank_by_volume = 1")["rows"][0]
assert (row[0], int(row[1])) == (top_code, top_count) or int(row[1]) == top_count
print(f"OK  most frequent reason: {row[0]} with {row[1]} encounters = source")

OK  admissions and discharges: 114 each = inpatient stays in the source


OK  emergency visits: 127 = EMER encounters in the source


OK  average stay: 6.218 days = source


OK  most frequent reason: 431857002 with 498 encounters = source


### 30-day readmissions: an independent count
Plain Python, straight from the source files: for each patient, sort the inpatient stays by start time; a stay is a readmission when it
starts between the end of the previous stay and 30 days after it. The total must equal what the view reports.

In [15]:
stays = {}
for e in inpatient:
    patient = e["subject"]["reference"]
    stays.setdefault(patient, []).append((utc(e["period"]["start"]), utc(e["period"]["end"])))

expected_readmissions = 0
for patient_stays in stays.values():
    patient_stays.sort()
    for (_, previous_end), (start, _) in zip(patient_stays, patient_stays[1:], strict=False):   # consecutive pairs, so the second list is one shorter
        if previous_end <= start <= previous_end + timedelta(days=30):
            expected_readmissions += 1

admissions, readmissions = [int(x) for x in run_athena(
    f"SELECT SUM(inpatient_admissions), SUM(readmissions_30d) FROM {GOLD}.vw_readmission_30d")["rows"][0]]
assert admissions == len(inpatient)
assert readmissions == expected_readmissions, (readmissions, expected_readmissions)
print(f"OK  {admissions} inpatient stays, {readmissions} readmissions within 30 days "
      f"(rate {readmissions / admissions:.1%}) = independent count from the source")

OK  114 inpatient stays, 17 readmissions within 30 days (rate 14.9%) = independent count from the source


### A look at each view

In [16]:
def show(view, order_by, limit=5):
    result = run_athena(f"SELECT * FROM {GOLD}.{view} ORDER BY {order_by} LIMIT {limit}")
    print(f"--- {view}")
    print("   ", " | ".join(result["header"]))
    for row in result["rows"]:
        print("   ", " | ".join(str(value)[:34] for value in row))


show("vw_daily_admissions_discharges", "admissions DESC, date_key")
show("vw_avg_los_by_facility_month", "inpatient_stays DESC, month")
show("vw_readmission_30d", "readmissions_30d DESC, month")
show("vw_ed_volume_by_hour_dow", "emergency_visits DESC")
show("vw_top_reasons", "rank_by_volume")

--- vw_daily_admissions_discharges
    date_key | admissions | discharges
    2007-08-16 | 2 | 0
    2022-11-03 | 2 | 1
    2024-10-05 | 2 | 0
    2026-10-16 | 2 | 0
    1943-12-08 | 1 | 0


--- vw_avg_los_by_facility_month
    month | organization_id | facility_name | inpatient_stays | avg_los_days | median_los_days
    2022-11-01 | 74ab949d-17ac-3309-83a0-13b4405c66 | Fitchburg Outpatient Clinic | 3 | 1.7099807098765432 | 1.0
    2023-01-01 | 74ab949d-17ac-3309-83a0-13b4405c66 | Fitchburg Outpatient Clinic | 3 | 1.022800925925926 | 1.0
    2024-10-01 | 74ab949d-17ac-3309-83a0-13b4405c66 | Fitchburg Outpatient Clinic | 3 | 1.0193595679012344 | 1.0
    2023-11-01 | 74ab949d-17ac-3309-83a0-13b4405c66 | Fitchburg Outpatient Clinic | 2 | 4.1225810185185185 | 7.245162037037037
    2025-09-01 | 74ab949d-17ac-3309-83a0-13b4405c66 | Fitchburg Outpatient Clinic | 2 | 2.535347222222222 | 3.0438541666666667


--- vw_readmission_30d
    month | organization_id | facility_name | inpatient_admissions | readmissions_30d | readmission_rate
    2006-09-01 | d6600c68-8a26-3a88-86e2-99517365c1 | New Bedford Outpatient Clinic | 1 | 1 | 1.0
    2007-05-01 | d6600c68-8a26-3a88-86e2-99517365c1 | New Bedford Outpatient Clinic | 1 | 1 | 1.0
    2007-08-01 | d6600c68-8a26-3a88-86e2-99517365c1 | New Bedford Outpatient Clinic | 1 | 1 | 1.0
    2008-03-01 | d6600c68-8a26-3a88-86e2-99517365c1 | New Bedford Outpatient Clinic | 1 | 1 | 1.0
    2008-10-01 | d6600c68-8a26-3a88-86e2-99517365c1 | New Bedford Outpatient Clinic | 1 | 1 | 1.0


--- vw_ed_volume_by_hour_dow
    hour_of_day_utc | day_of_week | day_name | emergency_visits
    15 | 5 | Friday | 11
    2 | 3 | Wednesday | 5
    21 | 1 | Monday | 4
    21 | 6 | Saturday | 4
    21 | 3 | Wednesday | 3


--- vw_top_reasons
    reason_code | reason_display | encounters | patients | rank_by_volume
    431857002 | Chronic kidney disease stage 4 (di | 498 | 3 | 1
    46177005 | End-stage renal disease (disorder) | 320 | 2 | 2
    72892002 | Normal pregnancy (finding) | 201 | 18 | 3
    389095005 | Contraception care (regime/therapy | 193 | 33 | 4
    66383009 | Gingivitis (disorder) | 182 | 93 | 5


## What the gold layer is for

| Object | Used by |
|---|---|
| `dim_*` and `fact_encounter` | the star schema: one fact table (encounters) surrounded by dimensions (who, where, when) |
| The five views | the KPI notebook (Phase 8) and the Power BI report (Phase 13) |
| `merge_*.sql` | run by Step Functions after the Glue job, one `ingest_date` at a time (Phase 7) |

Notes: ages are calculated on the day the merge runs; all times and the hour-of-day KPI are in UTC; with only 4 batches the
readmission numbers are small, but they match an independent count exactly.

**Next:** Step Functions chains the Lambda, the Glue job and these merges into one pipeline.